## Home Sales Price Prediction — Feature Engineering

This notebook prepares the final modeling dataset following exploratory analysis.

Objectives:

- Define model input features
- Evaluate multicollinearity among numerical variables
- Encode categorical variables
- Construct the final modeling dataset

The resulting dataset will be exported for use in the modeling notebook.

##### Import Libraries and Data

This section imports the libraries required for feature engineering and loads the cleaned dataset generated from the exploratory analysis stage.

Key libraries used:

- **pandas / numpy** → data manipulation
- **matplotlib / seaborn** → exploratory diagnostics
- **statsmodels** → multicollinearity diagnostics (VIF)
- **sklearn** → encoding and modeling utilities

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data Transformations for pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Improve plot styling
sns.set()

# Show all columns during EDA
pd.set_option('display.max_columns', None)

In [2]:
# File path locations
from pathlib import Path

# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

In [3]:
# Load dataset generated from the exploratory analysis notebook
# Parquet format preserves schema and improves load performance
data = pd.read_parquet(CLEAN_DATA_DIR / "02_exploratory_analysis.parquet")
data.head()

,neighborhood,building_class_category,tax_class_at_present,block,lot,building_class_at_present,address,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date,build_age_yrs,BBL,latitude,longitude,nearest_station,distance_to_station,log_sale_price,log_gross_sqft,price_sqft,within_half_mi,log_dist_to_station,station_distance_bucket
0,bath_beach,one_family_dwellings,1,6364,72,A5,68 BAY 14TH STREET,11214,1,0,1,1950,972,1950,1,A5,860000,2025-11-24,76,3063640072,40.607931,-74.007289,153,0.291308,5.934498,2.987666,884.773663,1,-0.535648,0.25-0.5
1,bath_beach,one_family_dwellings,1,6372,39,A9,86-29 19 AVENUE,11214,1,0,1,2417,1944,1930,1,A9,1180000,2025-11-19,96,3063720039,40.605178,-74.000846,108,0.146919,6.071882,3.288696,606.995885,1,-0.832923,0-0.25
2,bath_beach,one_family_dwellings,1,6401,46,A1,144 BAY 17 STREET,11214,1,0,1,4833,2164,1925,1,A1,1300000,2025-01-13,101,3064010046,40.604878,-74.006706,153,0.336323,6.113943,3.335257,600.739372,1,-0.473244,0.25-0.5
3,bath_beach,one_family_dwellings,1,6431,15,A9,221 BAY 13 STREET,11214,1,0,1,1530,2484,1920,1,A9,900000,2025-10-20,106,3064310015,40.605037,-74.010865,153,0.519558,5.954243,3.395152,362.318841,0,-0.284366,0.5-1
4,bath_beach,one_family_dwellings,1,6431,56,A5,200 BAY 14 STREET,11214,1,0,1,2086,1458,1910,1,A5,930000,2025-02-05,116,3064310056,40.605227,-74.010127,153,0.478792,5.968483,3.163758,637.860082,1,-0.319853,0.25-0.5


In [4]:
data.shape

(5596, 30)

### Define Feature Groups

Features are grouped by type to simplify preprocessing and model pipeline construction.

Feature groups:

- **Numerical features**  
  Continuous variables representing property characteristics.

- **Binary features**  
  Indicator variables capturing specific property conditions.

- **Categorical features**  
  Discrete variables representing geographic or structural segmentation.

Separating features by type allows appropriate preprocessing to be applied during modeling.

In [5]:
# Numerical features
numerical_features = [
    "log_gross_sqft",
    "build_age_yrs", 
    "log_dist_to_station",
    "residential_units"
]

In [6]:
# Binary features
binary_features = [
    "within_half_mi"
]

In [7]:
# Categorical features
categorical_features = [
    "neighborhood", 
    "building_class_category"
]

### Multicollinearity Analysis (Variance Inflation Factor)

Highly correlated predictors can destabilize linear regression models and inflate coefficient variance.

To diagnose this, we compute **Variance Inflation Factor (VIF)** for numerical predictors.

Interpretation:

- VIF < 5 → acceptable
- VIF 5–10 → moderate correlation
- VIF > 10 → strong multicollinearity

Variables with excessive VIF are iteratively removed to improve model stability.

In [8]:
# Import VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [9]:
# Remove VIF Iteratively

def calculate_vif(df, features, thresh):
    X = df[features].copy()
    variables = X.columns.tolist()
    
    while True:
        vif_data = pd.DataFrame()
        vif_data["Feature"] = variables
        vif_data["VIF"] = [
            variance_inflation_factor(X[variables].values, i)
            for i in range(len(variables))
        ]
        
        max_vif = vif_data["VIF"].max()
        
        if max_vif > thresh and len(variables) > 1:
            drop_feature = vif_data.sort_values("VIF", ascending=False)["Feature"].iloc[0]
            print(f"Dropping '{drop_feature}' with VIF: {max_vif:.2f}")
            variables.remove(drop_feature)
        else:
            break
    
    print("\nFinal numeric features:")
    print(variables)
    
    return vif_data.sort_values("VIF", ascending=False)

# Usage
cleaned_df = calculate_vif(data, numerical_features, 10)
cleaned_df


Final numeric features:
['log_gross_sqft', 'build_age_yrs', 'log_dist_to_station', 'residential_units']


,Feature,VIF
0,log_gross_sqft,7.725068
1,build_age_yrs,5.620575
2,log_dist_to_station,3.100983
3,residential_units,1.919581


In [10]:
# Remove "residential_units" and re-run VIF
numerical_features_reduced = [
    "log_gross_sqft",
    "build_age_yrs", 
    "log_dist_to_station"
]

cleaned_df = calculate_vif(data, numerical_features_reduced, 10)
cleaned_df


Final numeric features:
['log_gross_sqft', 'build_age_yrs', 'log_dist_to_station']


,Feature,VIF
0,log_gross_sqft,6.735788
1,build_age_yrs,5.554336
2,log_dist_to_station,3.050318


### Updated Feature Set

Following multicollinearity diagnostics, the numerical feature set was refined to remove highly correlated predictors.

The final feature set combines:

- binary features
- categorical features
- reduced numerical feature set

These features will be used to construct the final modeling matrix.

In [11]:
# Define features
feature_cols = binary_features + categorical_features + numerical_features_reduced
X = data[feature_cols].copy()

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Confirm X is still a DataFrame and has the expected columns
assert hasattr(X, "columns"), "X is not a DataFrame anymore."
print("X columns at transform time:", list(X.columns))

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features_reduced),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_processed = preprocessor.fit_transform(X)
X_processed.shape

X columns at transform time: ['within_half_mi', 'neighborhood', 'building_class_category', 'log_gross_sqft', 'build_age_yrs', 'log_dist_to_station']


(5596, 63)

### Preprocessing Pipeline

Instead of manually encoding categorical variables, a preprocessing pipeline is used.

This ensures that the same transformations applied during training are also applied
during model inference.

The pipeline performs:

• One-hot encoding for categorical variables  
• Feature scaling for numerical variables

In [13]:
X_processed.shape

(5596, 63)

### Export Final Modeling Dataset

The fully engineered feature matrix is exported for use in the modeling notebook.

The exported dataset contains:

- encoded predictor variables
- log-transformed target variable

This ensures modeling experiments are reproducible and separated from feature engineering logic.

In [14]:
X_processed_df = pd.DataFrame(
    X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X.index
)

modeling_df = X_processed_df.copy()
modeling_df["log_sale_price"] = data.loc[X.index, "log_sale_price"].values

In [15]:
modeling_df.to_parquet(CLEAN_DATA_DIR / "03_feature_engineering.parquet", index=False)

### Feature Engineering Summary

The following preprocessing steps were applied to construct the modeling dataset:

1. Feature grouping into numerical, binary, and categorical variables
2. Multicollinearity diagnostics using VIF
3. Removal of highly correlated predictors
4. One-hot encoding of categorical variables
5. Construction of the final modeling matrix

The resulting dataset will be used in the modeling stage to evaluate multiple regression algorithms.

In [16]:
# Import `joblib` to save prepreprocessing pipeline
import joblib

# Fit preprocessing pipeline
X_processed = preprocessor.fit_transform(X)

joblib.dump(
    preprocessor,
    MODELS_DIR / "preprocessing_pipeline.joblib"
)

['C:\\Users\\jac67\\Documents\\Data and Analytics\\Python\\brooklyn-home-sales\\models\\preprocessing_pipeline.joblib']